# The goal is to collect all the data from the different years and compare the best algos from each year to determine the best of the best : BBOB- biobj test Suite

# 1) Collecting the data from the different years and making one single file with all the best algos from all the different years.

We first build a "table" where we will agregate all the data from all of the different years.

In [10]:
import pandas as pd
from pathlib import Path

# === 1. Load all yearly CSVs from the "results" folder ===
folder = Path("results")
all_files = sorted(folder.glob("bbob-biobj_*.csv"))

dfs = []
for f in all_files:
    year = int(f.stem.split("_")[-1])
    df = pd.read_csv(f)
    df["year"] = year
    dfs.append(df)

# Merge everything into a single DataFrame
df_all = pd.concat(dfs, ignore_index=True)

# === 2. Find, for each (dim, func, target), the entry with smallest ERT ===
idx = df_all.groupby(["dimension", "function_id", "target"])["best_ERT"].idxmin()
df_best_overall = df_all.loc[idx, ["dimension", "function_id", "year", "best_algorithm", "target", "best_ERT"]]

# Sort nicely
df_best_overall = df_best_overall.sort_values(["dimension", "function_id", "target"]).reset_index(drop=True)

# === 3. Display result ===
print(" Global best algorithm across all years (smallest ERT for each dim/function/target):\n")
print(df_best_overall.head(20))

# Optionally save to file
df_best_overall.to_csv("results/global_best_algos_bbob-biobj.csv", index=False)


 Global best algorithm across all years (smallest ERT for each dim/function/target):

    dimension  function_id  year                    best_algorithm  \
0           2            1  2016                               NaN   
1           2            1  2016      SMS-EMOA-DE_Auger_bbob-biobj   
2           2            1  2016  HMO-CMA-ES_Loshchilov_bbob-biobj   
3           2            1  2021          DMS_Brockhoff_bbob-biobj   
4           2            1  2022                     K-RVEA_Tanabe   
5           2            2  2016                               NaN   
6           2            2  2016    UP-MO-CMA-ES_Krause_bbob-biobj   
7           2            2  2016  HMO-CMA-ES_Loshchilov_bbob-biobj   
8           2            2  2022                        TPB_Tanabe   
9           2            2  2021          DMS_Brockhoff_bbob-biobj   
10          2            3  2016                               NaN   
11          2            3  2016    UP-MO-CMA-ES_Krause_bbob-biobj   
12  

# 2) General table summing up nice stats about the best algos

- Where the best algos are located (year)
- Which algorithms are the dominant winners (counter)

This wil be used to check the results especially for the count.

In [11]:
# Count wins per year
print(df_best_overall["year"].value_counts())

# Count wins per algorithm
print(df_best_overall["best_algorithm"].value_counts().head(10))


2019    1396
2016    1155
2022     189
2021      20
Name: year, dtype: int64
HMO-CMA-ES_Loshchilov_bbob-biobj       487
GDE3-platypus_Brockhoff_bbob-biobj     188
SPEA2-platypus_Brockhoff_bbob-biobj    179
UP-MO-CMA-ES_Krause_bbob-biobj         163
RM-MEDA_Auger_bbob-biobj               111
TPB_Tanabe                              90
IBEA-platypus_Brockhoff_bbob-biobj      75
K-RVEA_Tanabe                           58
MOTPE_Tanabe                            41
COMO-316_dufosse_bbob-biobj             40
Name: best_algorithm, dtype: int64


The table bellow repertoriates the dimension for which the algorithms are best performing. 

In [12]:
# === 4. Aggregate: count how many times each algorithm was best per dimension ===

# Count how many times each algorithm appears as best within each dimension
algo_counts = (
    df_best_overall
    .groupby(["dimension", "best_algorithm"])
    .size()  # count occurrences
    .reset_index(name="count")
)

# Sort within each dimension by count descending
algo_counts = (
    algo_counts
    .sort_values(["dimension", "count"], ascending=[True, False])
)

# For convenience: for each dimension, keep rank of algorithms by count
algo_counts["rank"] = algo_counts.groupby("dimension")["count"].rank(method="first", ascending=False)


pivot_table = algo_counts.pivot(index="best_algorithm", columns="dimension", values="count").fillna(0).astype(int)
pivot_table["Total"] = pivot_table.sum(axis=1)
pivot_table = pivot_table.sort_values("Total", ascending=False)

print("\n Overall frequency of being best by algorithm and dimension (top 10):\n")
print(pivot_table.head(10))



 Overall frequency of being best by algorithm and dimension (top 10):

dimension                             2   3   5   10   20   40  Total
best_algorithm                                                       
HMO-CMA-ES_Loshchilov_bbob-biobj     50  82  99  125  131    0    487
GDE3-platypus_Brockhoff_bbob-biobj   11  15  17   17   19  109    188
SPEA2-platypus_Brockhoff_bbob-biobj  53  38  30   18   14   26    179
UP-MO-CMA-ES_Krause_bbob-biobj       51  33  38   20   21    0    163
RM-MEDA_Auger_bbob-biobj             14  14   3    2    1   77    111
TPB_Tanabe                           13  18  19   19   21    0     90
IBEA-platypus_Brockhoff_bbob-biobj   16  16  14   18   11    0     75
K-RVEA_Tanabe                        26  20  10    1    1    0     58
MOTPE_Tanabe                         15  13  10    3    0    0     41
COMO-316_dufosse_bbob-biobj           1   4  13   15    7    0     40


# 3) Results : 2 different types of tables. 
The goal is to nicely present our results in tables summing up which are the best algorithms over all. 
The definition of “best algorithm” is NOT different between the tables. What is different is the DATA they aggregate.


- We are going to present our results in 2 different ways. A first one being getting the best algo over all dimensions, function AND aggregating over every target precision. 

- A second way is to determine the best algo over all dimensions, functions BUT only for a specific target, and aggregate only on that target. 

## 3.1) Aggregating over every target 



### 3.1.1) Table of the best algos
Here we are building a table that sums up the best algorithms for each dimension: there are 6 different dimensions. We can also see the count for how many times that algorithm was the best in that specific dimension 

The columns of this table are the different dimmension, and the lines of the table represent the ranking of the best algorithms. In the first line we will have the best algorithm for each dimension. The second line will represnt the 2nd best algorithms for each dimension etc... 

In the sence that we are counting wins across all targets at once. Then rank algorithms per dimension across this big mixture.
i.e. “Across ALL target precisions, which algorithm wins the most often in each dimension?”

In [14]:
# === 4. Aggregate: count how many times each algorithm was best per dimension ===
algo_counts = (
    df_best_overall
    .groupby(["dimension", "best_algorithm"])
    .size()
    .reset_index(name="count")
)

# Sort within each dimension by how often each algo was best
algo_counts = algo_counts.sort_values(["dimension", "count"], ascending=[True, False])

# Add ranking per dimension
algo_counts["rank"] = (
    algo_counts
    .groupby("dimension")["count"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# === NEW PART: build a tuple column (algo, count) ===
algo_counts["algo_tuple"] = list(zip(algo_counts["best_algorithm"], algo_counts["count"]))

# === 5. Pivot: rows = rank, columns = dimension, values = (algo_name, count) tuple ===
algo_ranking_table = algo_counts.pivot(index="rank", columns="dimension", values="algo_tuple")

# Sort dimensions in ascending order
algo_ranking_table = algo_ranking_table.reindex(sorted(algo_ranking_table.columns), axis=1)

# === 6. Display neatly ===
print("\n Ranking of best algorithms per dimension (with counts):\n")
from IPython.display import display
display(algo_ranking_table.head(10))  # show top 10 ranks



 Ranking of best algorithms per dimension (with counts):



dimension,2,3,5,10,20,40
rank,,,,,,
1,"(SPEA2-platypus_Brockhoff_bbob-biobj, 53)","(HMO-CMA-ES_Loshchilov_bbob-biobj, 82)","(HMO-CMA-ES_Loshchilov_bbob-biobj, 99)","(HMO-CMA-ES_Loshchilov_bbob-biobj, 125)","(HMO-CMA-ES_Loshchilov_bbob-biobj, 131)","(GDE3-platypus_Brockhoff_bbob-biobj, 109)"
2,"(UP-MO-CMA-ES_Krause_bbob-biobj, 51)","(SPEA2-platypus_Brockhoff_bbob-biobj, 38)","(UP-MO-CMA-ES_Krause_bbob-biobj, 38)","(UP-MO-CMA-ES_Krause_bbob-biobj, 20)","(TPB_Tanabe, 21)","(RM-MEDA_Auger_bbob-biobj, 77)"
3,"(HMO-CMA-ES_Loshchilov_bbob-biobj, 50)","(UP-MO-CMA-ES_Krause_bbob-biobj, 33)","(SPEA2-platypus_Brockhoff_bbob-biobj, 30)","(TPB_Tanabe, 19)","(UP-MO-CMA-ES_Krause_bbob-biobj, 21)","(SPEA2-platypus_Brockhoff_bbob-biobj, 26)"
4,"(K-RVEA_Tanabe, 26)","(K-RVEA_Tanabe, 20)","(TPB_Tanabe, 19)","(IBEA-platypus_Brockhoff_bbob-biobj, 18)","(GDE3-platypus_Brockhoff_bbob-biobj, 19)","(MOEAD-platypus_Brockhoff_bbob-biobj, 9)"
5,"(IBEA-platypus_Brockhoff_bbob-biobj, 16)","(TPB_Tanabe, 18)","(GDE3-platypus_Brockhoff_bbob-biobj, 17)","(SPEA2-platypus_Brockhoff_bbob-biobj, 18)","(SPEA2-platypus_Brockhoff_bbob-biobj, 14)","(SMS-EMOA-DE_Auger_bbob-biobj, 1)"
6,"(MOTPE_Tanabe, 15)","(IBEA-platypus_Brockhoff_bbob-biobj, 16)","(IBEA-platypus_Brockhoff_bbob-biobj, 14)","(GDE3-platypus_Brockhoff_bbob-biobj, 17)","(IBEA-platypus_Brockhoff_bbob-biobj, 11)",NaN
7,"(MO-DIRECT-HV-Rank_Wong_bbob-biobj, 14)","(GDE3-platypus_Brockhoff_bbob-biobj, 15)","(COMO-316_dufosse_bbob-biobj, 13)","(COMO-316_dufosse_bbob-biobj, 15)","(COMO-10_dufosse_bbob-biobj, 7)",NaN
8,"(RM-MEDA_Auger_bbob-biobj, 14)","(RM-MEDA_Auger_bbob-biobj, 14)","(COMO-100_dufosse_bbob-biobj, 12)","(COMO-100_dufosse_bbob-biobj, 7)","(COMO-316_dufosse_bbob-biobj, 7)",NaN
9,"(TPB_Tanabe, 13)","(MOTPE_Tanabe, 13)","(K-RVEA_Tanabe, 10)","(COMO-3_dufosse_bbob-biobj, 5)","(COMO-100_dufosse_bbob-biobj, 5)",NaN


We have noticed that the best algorithms come as batches of 6 algorithms per line. So hence when we are asking for the N best algorithms, where N<6, it is complicated to choose which algorithms to return. This is why we will set a minimum of 6 best algorithms, one for each dimension. Now we may be asked for more than 6 best algrithms. 


For example let's say we are asking for the 8 best algorithms. In this case we will go through the following lines of the table. There are several cases we need to consider to choose these 8 best algorithms.

- One scenario, is that in the following line, there are the names of algorithms that were already mentioned in the first line. In this case we don't add their name again. 
- The second scenario is we are in fact not able to return the names of 8 algorithms, because all the algorithms have performed equally for their respective dimension. Instead of choosing only 8, we will return the names of all the algorithms that are mentioned and are on the same line. We also notify that we are not returning 8 algorithms, but a bit more, here 10.
- A further step would be to really restrict the number to 8. This means we need to find a way to classify the algorithms that are on the same line. An easy way is to select the one that has the highest occurence of "best" algorithm. 

### 3.1.2) List of the best algos according to the table

Here our goal is to return N number of best algo depending on how many best algo the user is asking for. We will treat every algo on a same row equally, not one being better than the other. 

In [17]:
def get_top_algorithms(algo_ranking_table, N_min):
    """
    Collect algorithms from top ranks (rows) across all dimensions
    until at least N_min unique algorithms are found.
    """
    seen_algos = set()
    row_idx = 0

    # Keep adding rows until we have at least N_min unique algorithms
    while len(seen_algos) < N_min and row_idx < len(algo_ranking_table):
        row_algos = algo_ranking_table.iloc[row_idx].dropna().unique()

        # Extract only the algorithm names (first element of tuple)
        algo_names = {algo_tuple[0] for algo_tuple in row_algos}

        # Add these clean names to the seen set
        seen_algos.update(algo_names)
        row_idx += 1

    algo_list = sorted(seen_algos)

    print(f"\n Requested {N_min} best algorithms.")
    print(f" Returning {len(algo_list)} unique algorithms (reached rank {row_idx}).\n")
    print(" List of selected algorithms:\n")
    for algo in algo_list:
        print(f" - {algo}")

    return algo_list


# === Interactive part ===
try:
    # Ask user for number of desired algorithms
    N_input = int(input("How many best algorithms do you want? "))
    if N_input < 1:
        print(" Minimum number of best algorithms is 1. Using N=1.")
        N_input = 1

    # Compute and display
    best_algos = get_top_algorithms(algo_ranking_table, N_min=N_input)

except ValueError:
    print("Invalid input. Please enter an integer number (e.g., 6 or 7).")


How many best algorithms do you want? 1

 Requested 1 best algorithms.
 Returning 4 unique algorithms (reached rank 1).

 List of selected algorithms:

 - COMO-316_dufosse_bbob-biobj
 - GDE3-platypus_Brockhoff_bbob-biobj
 - HMO-CMA-ES_Loshchilov_bbob-biobj
 - UP-MO-CMA-ES_Krause_bbob-biobj


## 3.2) Choice of a specific target precision for the best algo.

### 3.2.1) Table dpending on a specific target precision


- We filter by ONE target precision of our choice.

- Count wins only for that target.

- Rank algos per dimension based only on those wins. 

i.e. “For target = X, which algorithm wins the most functions in each dimension?”

In [20]:
import pandas as pd

# --- assuming df_best_overall is already built as before ---
# columns: ["dimension", "function_id", "year", "best_algorithm", "target", "best_ERT"]


def make_dimension_ranking_table_with_counts(df_best_overall, target):
    """
    For a given target:
    - Count, for each dimension, how many times each algorithm was best.
    - Rank algorithms within each dimension by this count.
    - Pivot into a table: rows = rank, columns = dimension,
      values = (algorithm, count) tuples.
    """
    # Filter by target
    df_t = df_best_overall[df_best_overall["target"] == target].copy()
    if df_t.empty:
        raise ValueError(f"No data found for target = {target}")

    # Aggregate: count how many times each algorithm is best in each dimension
    algo_counts = (
        df_t.groupby(["dimension", "best_algorithm"])
        .size()
        .reset_index(name="count")
    )

    # Sort inside each dimension
    algo_counts = algo_counts.sort_values(
        ["dimension", "count"], ascending=[True, False]
    )

    # Rank per dimension
    algo_counts["rank"] = (
        algo_counts.groupby("dimension")["count"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    # Build tuple column
    algo_counts["algo_tuple"] = list(
        zip(algo_counts["best_algorithm"], algo_counts["count"])
    )

    # Pivot: rows = rank, columns = dimension, values = tuple
    ranking_table = algo_counts.pivot(
        index="rank", columns="dimension", values="algo_tuple"
    )

    # Sort dimensions
    ranking_table = ranking_table.reindex(
        sorted(ranking_table.columns), axis=1
    )

    return ranking_table


# === Interactive part: only ask for target ===
try:
    print("Available targets:", df_best_overall["target"].unique())

    # User input
    target_input = float(input("\nSelect a target precision (e.g., 1e-8 or 1e-5): "))

    # Build the table
    algo_ranking_table = make_dimension_ranking_table_with_counts(
        df_best_overall,
        target_input
    )

    print(f"\nRanking table built for target = {target_input}")
    from IPython.display import display
    display(algo_ranking_table.head(3))

except ValueError:
    print("Invalid target input. Please enter a numeric target.")


Available targets: [1.e-08 1.e-05 1.e-03 1.e-02 1.e-01]

Select a target precision (e.g., 1e-8 or 1e-5): 1.e-03

Ranking table built for target = 0.001


dimension,2,3,5,10,20,40
rank,,,,,,
1,"(HMO-CMA-ES_Loshchilov_bbob-biobj, 14)","(HMO-CMA-ES_Loshchilov_bbob-biobj, 24)","(HMO-CMA-ES_Loshchilov_bbob-biobj, 32)","(HMO-CMA-ES_Loshchilov_bbob-biobj, 43)","(HMO-CMA-ES_Loshchilov_bbob-biobj, 45)","(RM-MEDA_Auger_bbob-biobj, 24)"
2,"(RM-MEDA_Auger_bbob-biobj, 14)","(RM-MEDA_Auger_bbob-biobj, 6)","(COMO-316_dufosse_bbob-biobj, 7)","(COMO-316_dufosse_bbob-biobj, 8)","(COMO-316_dufosse_bbob-biobj, 3)","(GDE3-platypus_Brockhoff_bbob-biobj, 13)"
3,"(GDE3-platypus_Brockhoff_bbob-biobj, 7)","(GDE3-platypus_Brockhoff_bbob-biobj, 5)","(UP-MO-CMA-ES_Krause_bbob-biobj, 5)","(GDE3-platypus_Brockhoff_bbob-biobj, 2)","(UP-MO-CMA-ES_Krause_bbob-biobj, 3)","(SPEA2-platypus_Brockhoff_bbob-biobj, 4)"


# 4) Plotting the results

In [5]:
import cocopp
cocopp.main(['SPEA2-platypus_Brockhoff_bbob-biobj','HMO-CMA-ES_Loshchilov_bbob-biobj', 'GDE3-platypus_Brockhoff_bbob-biobj'])

Post-processing (2+)
  Using 3 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\SPEA2-platypus_Brockhoff_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\HMO-CMA-ES_Loshchilov_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\GDE3-platypus_Brockhoff_bbob-biobj.tgz

Post-processing (2+)
  loading data...
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\SPEA2-platypus_Brockhoff_bbob-biobj.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\HMO-CMA-ES_Loshchilov_bbob-biobj.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\GDE3-platypus_Brockhoff_bbob-biobj.tgz


C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pproc.py:3895: UserWarning:  Reference values for the algorithm 'C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\HMO-CMA-ES_Loshchilov_bbob-biobj.tgz' are different!
  warnings.warn(" Reference values for the algorithm '%s' are different!" % alg)
C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pproc.py:3895: UserWarning:  Reference values for the algorithm 'C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\GDE3-platypus_Brockhoff_bbob-biobj.tgz' are different!
  warnings.warn(" Reference values for the algorithm '%s' are different!" % alg)
C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\testbedsettings.py:126: UserWarning:  Reference values for the algorithm 'HMO-CMA-ES_Loshchilov_bbob-biobj' are different from the algorithm 'SPEA2-platypus_Brockhoff_bbob-biobj'
  warnings.warn(" Reference values for the algorithm '%s' are different from the algorithm '%s'"


  Will generate output data in folder ppdata\biobj-ext_SPEA2_HMO-C_GDE3-_112701h0106
    this might take several minutes.
ECDF graphs per noise group...
Loading best algorithm data from refalgs/best2016-bbob-biobj.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2016-bbob-biobj.tar.gz


C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\testbedsettings.py:126: UserWarning:  Reference values for the algorithm 'refalgs/best2016-bbob-biobj' are different from the algorithm 'SPEA2-platypus_Brockhoff_bbob-biobj'
  warnings.warn(" Reference values for the algorithm '%s' are different from the algorithm '%s'"


  done (Thu Nov 27 01:01:08 2025).
  done (Thu Nov 27 01:01:17 2025).
ECDF graphs per function group...
  done (Thu Nov 27 01:02:40 2025).
ECDF graphs per function...
  done (Thu Nov 27 01:07:35 2025).
Generating comparison tables...
  done (Thu Nov 27 01:08:00 2025).
Scaling figures...
  done (Thu Nov 27 01:08:57 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\biobj-ext_SPEA2_HMO-C_GDE3-_112701h0106
Setting changes in `cocopp.genericsettings` compared to default:
    simulated_runlength_bootstrap_sample_size: from 30 to 10.098990100989901
    foreground_algorithm_list: from [] to ['C:\\Users\\elsaf\\Ap...
ALL done (Thu Nov 27 01:08:57 2025).


DictAlg([(('SPEA2-platypus_Brockhoff_bbob-biobj', ''),
          [DataSet(SPEA2-platypus_Brockhoff_bbob-biobj on f1 2-D),
           DataSet(SPEA2-platypus_Brockhoff_bbob-biobj on f2 2-D),
           DataSet(SPEA2-platypus_Brockhoff_bbob-biobj on f3 2-D),
           DataSet(SPEA2-platypus_Brockhoff_bbob-biobj on f4 2-D),
           DataSet(SPEA2-platypus_Brockhoff_bbob-biobj on f5 2-D),
           DataSet(SPEA2-platypus_Brockhoff_bbob-biobj on f6 2-D),
           DataSet(SPEA2-platypus_Brockhoff_bbob-biobj on f7 2-D),
           DataSet(SPEA2-platypus_Brockhoff_bbob-biobj on f8 2-D),
           DataSet(SPEA2-platypus_Brockhoff_bbob-biobj on f9 2-D),
           DataSet(SPEA2-platypus_Brockhoff_bbob-biobj on f10 2-D),
           DataSet(SPEA2-platypus_Brockhoff_bbob-biobj on f11 2-D),
           DataSet(SPEA2-platypus_Brockhoff_bbob-biobj on f12 2-D),
           DataSet(SPEA2-platypus_Brockhoff_bbob-biobj on f13 2-D),
           DataSet(SPEA2-platypus_Brockhoff_bbob-biobj on f14 2-D),
  

Asboslute best results for a given target

In [6]:
cocopp.main(['UP-MO-CMA-ES_Krause_bbob-biobj','HMO-CMA-ES_Loshchilov_bbob-biobj', 'COMO-316_dufosse_bbob-biobj', 'GDE3-platypus_Brockhoff_bbob-biobj'])

Post-processing (2+)
  Using 4 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\UP-MO-CMA-ES_Krause_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\HMO-CMA-ES_Loshchilov_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\COMO-316_dufosse_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\GDE3-platypus_Brockhoff_bbob-biobj.tgz

Post-processing (2+)
  loading data...
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\UP-MO-CMA-ES_Krause_bbob-biobj.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\HMO-CMA-ES_Loshchilov_bbob-biobj.tgz
  Data consistent according to consistency_check() in pproc.DataSet
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\COMO-316_dufosse_bbob-biobj.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cach

C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pproc.py:3895: UserWarning:  Reference values for the algorithm 'C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\GDE3-platypus_Brockhoff_bbob-biobj.tgz' are different!
  warnings.warn(" Reference values for the algorithm '%s' are different!" % alg)
C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\testbedsettings.py:126: UserWarning:  Reference values for the algorithm 'GDE3-platypus_Brockhoff_bbob-biobj' are different from the algorithm 'UP-MO-CMA-ES_Krause_bbob-biobj'
  warnings.warn(" Reference values for the algorithm '%s' are different from the algorithm '%s'"


  Will generate output data in folder ppdata\biobj_UP-MO_HMO-C_COMO-_GDE3-_112701h1116
    this might take several minutes.
ECDF graphs per noise group...
Loading best algorithm data from refalgs/best2016-bbob-biobj.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2016-bbob-biobj.tar.gz
  done (Thu Nov 27 01:11:19 2025).
  done (Thu Nov 27 01:11:29 2025).
ECDF graphs per function group...
  done (Thu Nov 27 01:12:49 2025).
ECDF graphs per function...
  done (Thu Nov 27 01:17:59 2025).
Generating comparison tables...
  done (Thu Nov 27 01:18:25 2025).
Scaling figures...
  done (Thu Nov 27 01:19:23 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\biobj_UP-MO_HMO-C_COMO-_GDE3-_112701h1116
Setting changes in `cocopp.genericsettings` compared to default:
    simulated_runlength_bootstrap_sample_size: from 30 to 10.098990100989901
    foreground_algorithm_list: from [] to ['C:\\Users\\elsaf\\Ap...
ALL done (Thu Nov 27 01:19:23

DictAlg([(('UP-MO-CMA-ES_Krause_bbob-biobj', ''),
          [DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f1 2-D),
           DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f2 2-D),
           DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f3 2-D),
           DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f4 2-D),
           DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f5 2-D),
           DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f6 2-D),
           DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f7 2-D),
           DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f8 2-D),
           DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f9 2-D),
           DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f10 2-D),
           DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f11 2-D),
           DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f12 2-D),
           DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f13 2-D),
           DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f14 2-D),
           DataSet(UP-MO-CMA-ES_Krause_bbob-biobj on f15 2-D),
           Dat